# Lecture 8: Sensitivities: the state transition matrix and its jumps

**The challenge.** Every solver in this course adjusts unknowns (initial
costates, multipliers, a final time, junction times) until the boundary
conditions hold. Newton's method does that fastest, but it needs the
*derivative* of the end conditions with respect to the unknowns: how $z(t_f)$
moves when $z(t_0)$ moves. Finite differences give it, at a price. Each
unknown costs another integration; the step size must be tuned between
truncation error and round-off; and across a switch the perturbed trajectory
switches at a slightly different time, so the difference quotient mixes two
different arcs. Near a jump, finite differences are both expensive and
unreliable.

This lecture computes the derivative exactly, by integrating its own
differential equation alongside the trajectory: the **state transition
matrix**. Across a switch it must jump, by the saltation matrix sketched in
Lecture 3. We also differentiate with respect to parameters and the final
time, which continuation (Lecture 10) and free-time problems need.

**After this lecture you can**

- derive and integrate the variational equation, and say what the STM's
  blocks mean;
- explain and check the saltation jump at a switch, and what goes wrong
  without it;
- get $\partial z(t_f)/\partial z_0$, $\partial z(t_f)/\partial p$ and
  $\partial z(t_f)/\partial t_f$ from `propagate`, and the residual's
  derivatives from `residual_jacobian`.

## 1. Poincaré's error

The variational equation is older than control theory by half a century. In
1885 King Oscar II of Sweden offered a prize for progress on the stability of
the solar system, the $n$-body problem. Henri Poincaré won it in 1889 with a
memoir on the three-body problem, in which he linearized the equations of
motion around a periodic orbit and studied how small deviations grow: the
variational equation, and the matrix that propagates its solutions.

While the memoir was being printed in *Acta Mathematica*, the editor's
assistant Lars Edvard Phragmén queried a point. Poincaré found that he had
made a serious error. Certain solutions he had assumed well-behaved were in
fact tangled in an infinitely complicated way. The printed copies were
recalled and destroyed at Poincaré's expense, which cost him more than the
prize was worth, and he rewrote the memoir. The corrected version contained
the first description of what is now called chaos: trajectories whose
sensitivity to initial conditions, measured by exactly the matrix of this
lecture, grows without bound.

The name for the jump at a switch is much younger. Mark Aizerman and Felix
Gantmacher, in Moscow, worked out in 1958 how the variational equation
behaves across a discontinuity of the right-hand side, the **saltation
matrix**, from the Latin *saltare*, to jump.

## 2. The variational equation

Perturb the initial point, $z_0 \to z_0 + \delta z_0$. To first order the
perturbation obeys the linearization of $\dot z = F(t, z)$ along the
trajectory,
$$
\dot{\delta z} = A(t)\,\delta z, \qquad A(t) = \frac{\partial F}{\partial z}\Big|_{z(t)},
$$
so $\delta z(t) = \Phi(t)\,\delta z_0$, where the **state transition matrix**
solves
$$
\dot\Phi = A(t)\,\Phi, \qquad \Phi(t_0) = I .
$$
For $z = (x, \lambda)$, $\Phi$ has four blocks. The one shooting needs most is
$\Phi_{x\lambda} = \partial x(t_f)/\partial\lambda_0$: how the final state
responds to the initial costate. Two more derivatives complete the set:

- **parameters**: $\Psi = \partial z/\partial p$ solves
  $\dot\Psi = A\Psi + \partial F/\partial p$ with $\Psi(t_0) = 0$;
- **the final time**: $\partial z(t_f)/\partial t_f = F(t_f, z(t_f))$, the
  velocity at the end.

$A$ and $\partial F/\partial p$ are generated exactly by SymPy (the
`jacobian` and `dfdp` kernels), so the STM is exact up to integration error.
For $n$ states, integrating it adds $4n^2$ equations to the $2n$ of the
extremal.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from auto_optimal_control import examples
from auto_optimal_control.build import build_kernels
from auto_optimal_control.pontryagin import derive

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"

k = build_kernels(derive(examples.planar_low_thrust_fuel()))
p = np.array([1.0, 0.05, 1.0, 1.0, 1.0, 1.5, 6.0])
z0 = np.array([1.0, 0.0, 0.0, 1.0, 1.0, 0.2, 0.1, 0.1, 0.95, 0.0])
opts = dict(rtol=1e-12, atol=1e-12)

r = k.propagate(0.0, z0, 6.0, p, stm=True, param_sensitivity=True, **opts)
print("switches:", np.round(r["switch_times"], 6))
print("STM", r["stm"].shape, " dz/dp", r["dz_dp"].shape, " dz/dt_f", r["dz_dt1"].shape)
print("dz/dt_f equals F at the end:",
      np.allclose(r["dz_dt1"], k.rhs_arc(6.0, r["z"], p, r["arc"])))

## 3. Checked against finite differences

Central differences of the final state, one column per component of $z_0$,
should match the STM to the accuracy of the differences:

In [ ]:
def z_end(z_start, params=p, t_end=6.0):
    return k.propagate(0.0, z_start, t_end, params, **opts)["z"]


h = 1e-6
start = time.perf_counter()
fd = np.column_stack([(z_end(z0 + h * e) - z_end(z0 - h * e)) / (2 * h) for e in np.eye(10)])
t_fd = time.perf_counter() - start
start = time.perf_counter()
Phi = k.propagate(0.0, z0, 6.0, p, stm=True, **opts)["stm"]
t_stm = time.perf_counter() - start
print(f"max |Phi - finite differences| = {np.abs(Phi - fd).max():.1e}"
      f"   (largest entry {np.abs(Phi).max():.1e})")
print(f"time: STM {1e3 * t_stm:.2f} ms, finite differences {1e3 * t_fd:.2f} ms (20 integrations)")

fdp = np.column_stack([(z_end(z0, p + h * e) - z_end(z0, p - h * e)) / (2 * h) for e in np.eye(7)])
print(f"max |dz/dp - finite differences| = {np.abs(r['dz_dp'] - fdp).max():.1e}")

The STM agrees with the differences to about $10^{-8}$, the accuracy of a
central difference with this step. It is exact up to the integration
tolerance, and it costs one integration rather than twenty. The difference
in cost grows with the number of states.

## 4. The jump at a switch

Lecture 3 derived the jump. At a switch at time $t_s$, where the arc changes
from $F^-$ to $F^+$, a perturbed trajectory switches at a slightly different
time, and the perturbation is mapped by
$$
\Phi^+ = \Big( I + \frac{(F^+ - F^-)\,\nabla S^\top}{\dot S} \Big)\,\Phi^-,
\qquad \dot S = \partial_t S + \nabla S \cdot F^- .
$$
The package applies it at every switch, inside the hybrid integration, as the
guard's reset map (Lecture 7). The `stm_jump` kernel exposes it: given
$y = [z, \Phi]$ just before switch $k$ and the arc there, it returns $y$ just
after. To see that the jump matters, integrate to just before the first
switch, then from just after it to the end, and chain the two STMs with and
without the jump.

In [ ]:
ts = r["switch_times"][0]
before = k.propagate(0.0, z0, ts, p, stm=True, **opts)
y_before = np.concatenate([before["z"], before["stm"].ravel()])
y_after = k.stm_jump(0, ts, y_before, p, before["arc"])
after = k.propagate(ts, before["z"], 6.0, p, stm=True, arc=1 - before["arc"], **opts)
with_jump = after["stm"] @ y_after[10:].reshape(10, 10)
without_jump = after["stm"] @ before["stm"]
print(f"with the jump:    max |Phi - FD| = {np.abs(with_jump - fd).max():.1e}")
print(f"without the jump: max |Phi - FD| = {np.abs(without_jump - fd).max():.1e}")

Without the jump the "derivative" is wrong at order one, and a Newton
iteration built on it would wander. Notice also that the jump needs
$\dot S \ne 0$: at a tangential switch it is undefined, which is why
Lecture 7's status `-3` exists.

## 5. Structure: the flow is symplectic

Lecture 3 showed that the canonical equations have the Hamiltonian form
$\dot z = J\nabla H$ with $J = \begin{pmatrix} 0 & I \\ -I & 0 \end{pmatrix}$.
A Hamiltonian flow preserves the symplectic form: $\Phi^\top J \Phi = J$, so in
particular $\det\Phi = 1$. That holds here even across the switches, and it
is a cheap check of an STM:

In [ ]:
n = 5
Jm = np.block([[np.zeros((n, n)), np.eye(n)], [-np.eye(n), np.zeros((n, n))]])
print(f"max |Phi' J Phi - J| = {np.abs(Phi.T @ Jm @ Phi - Jm).max():.1e},  det Phi = {np.linalg.det(Phi):.12f}")

Symplecticity also explains why shooting is hard. A symplectic matrix's
singular values come in reciprocal pairs, $\sigma$ and $1/\sigma$: if some
directions are stretched by $e^{\mu t}$, others are squeezed by $e^{-\mu t}$.
The costate equations run the linearization "backwards"
($\dot\lambda = -A^\top\lambda$ near a trajectory), so a state direction that
the dynamics stabilize is paired with a costate direction that they blow up.
The spread between the largest and smallest singular values, and with it the
conditioning of Newton's problem, grows with the length of the trajectory:

In [ ]:
for t_end in [1.0, 2.0, 4.0, 6.0]:
    P = k.propagate(0.0, z0, t_end, p, stm=True, **opts)["stm"]
    sv = np.linalg.svd(P, compute_uv=False)
    print(f"t_f = {t_end:.0f}: largest sigma {sv[0]:8.2e}, smallest {sv[-1]:8.2e}, "
          f"product {sv[0] * sv[-1]:.6f}")

In this orbit problem the growth is mild. In unstable dynamics, like the
cart-pole's swinging pendulum in Lecture 9, it is exponential, and it is the
main reason good costate guesses matter.

## 6. From the STM to the residual's Jacobian

Shooting (Lecture 9) solves boundary conditions $R(z_0, z_f, \nu, t_f) = 0$.
The generated `residual` evaluates them and `residual_jacobian` returns
their partial derivatives with respect to every argument. The chain rule
then gives the derivative with respect to the unknowns:
$$
\frac{dR}{dz_0} = R_{z_0} + R_{z_f}\,\Phi, \qquad
\frac{dR}{dt_f} = R_{t_f} + R_{z_f}\,F(t_f), \qquad
\frac{dR}{dp} = R_p + R_{z_f}\,\Psi .
$$

In [ ]:
nu = np.zeros(k.N_MULTIPLIERS)
R = k.residual(z0, r["z"], nu, 6.0, p)
J = k.residual_jacobian(z0, r["z"], nu, 6.0, p)
print("residuals:", len(R), " blocks:", {key: np.shape(v) for key, v in J.items()})
dR_dz0 = J["z0"] + J["zf"] @ r["stm"]
print("dR/dz0", dR_dz0.shape)

That product, restricted to the columns that are unknowns, is the exact
Newton Jacobian. Lecture 9 builds on it, and with junction times as extra
unknowns Lectures 12 and 14 chain the same pieces across several arcs.

## Summary

- The STM $\Phi$ solves $\dot\Phi = A\Phi$ along the extremal; with
  $\Psi = \partial z/\partial p$ and $\partial z/\partial t_f = F$ it gives
  every derivative shooting and continuation need, exactly.
- At a switch $\Phi$ jumps by the saltation matrix
  $I + (F^+ - F^-)\nabla S^\top / \dot S$; without it the derivative is wrong
  at order one.
- `propagate(..., stm=True, param_sensitivity=True)` returns `stm`, `dz_dp`
  and `dz_dt1`; `jacobian`, `dfdp` and `stm_jump` are the pieces;
  `residual` and `residual_jacobian` give the boundary conditions and their
  partial derivatives.
- The flow is symplectic, $\Phi^\top J\Phi = J$, which is both a check and the
  reason long shooting problems are ill-conditioned.

## Exercises

1. Repeat section 3 with $h = 10^{-3}$ and $h = 10^{-10}$. Which error
   dominates in each case?
2. Move the end time across the second switch ($t_f$ from 4.6 to 4.7). Plot
   one entry of $\Phi$ against $t_f$. Is it continuous? Is its derivative?
3. For the minimum-time double integrator, compute the saltation matrix by
   hand at the switch and compare with `stm_jump`.

## Further reading

- J. Barrow-Green, *Poincaré and the Three Body Problem* (AMS, 1997): the
  prize, the error and the recall.
- M. A. Aizerman and F. R. Gantmacher, "Determination of stability by
  linear approximation of a periodic solution of a system of differential
  equations with discontinuous right-hand sides", *Quarterly Journal of
  Mechanics and Applied Mathematics* 11, 1958.
- A. E. Bryson and Y.-C. Ho, *Applied Optimal Control*, section 7.3, on
  transition matrices for shooting.